# Demo - Shape Deformation for Bandsaw Cutting

This is a minimal setup to generate similar shapes for bandsaw cutting as in the paper (Sec. 4.3).

This notebook requires more dependencies than the `dvx-python` module. Take sure to install the dependencies:

````powershell
python -m venv .env
.env/Scripts/Activate.ps1 # depends on your os
pip install -r demos/requirements.txt
````

In [ ]:
import math
import torch
import tqdm
import dvx.torch as dvx
import arap # as-rigid-as-possible energy in torch
import util # utility functions to keep the notebook clean
import polyscope as ps

torch.manual_seed(0)

## Setup parameters

In [ ]:
### configuration
mesh_path = "./meshes/hand.obj" # path to input mesh
d = 128                          # voxel resolution
max_steps = 1500                 # maximum number of optimization steps
step = 0.001                     # learning rate
alpha = 0.005                    # weight of arap loss
eps=5e-4                         # converged if max vertex change is below eps

# load and normalize mesh
V, F = util.load_obj(mesh_path)
V -= V.mean(0)
V /= torch.linalg.vector_norm(V, dim=1).max() * 1.25 # we leave some space for deformation

## Loss definition and helper functions

In [ ]:
def loss_fn(U: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
  # deformation loss
  loss_deform = arap.ARAP_energy(U, V, F)

  # cut loss
  q = dvx.voxelize(d, U, F) # call the differentiable voxelizer
  q_cut = cut_from_volume(q)
  loss_bandsaw = 1./(d**3) * (q_cut - q).pow(2).sum() # additional volume produced by bandsaw

  loss = loss_bandsaw + alpha * loss_deform
  return loss, loss_bandsaw, loss_deform

# helper
def mellowmax(q: torch.Tensor, dim: int, w: float = 1000, keepdim=False) -> torch.Tensor:
  """ The mellowmax operation on tensor q along dimension dim with weight w.
  This is a smooth way for projecting a 3D volume to a 2D slice along dim.
  w -> 0 gives the mean and w -> inf gives max along dim.
  """
  m = q.max(dim, keepdim=True).values
  lse = torch.exp((q - m)*w).sum(dim=dim, keepdim=keepdim).log()
  result = m + (lse - math.log(q.size(dim)))/w
  return result

def cut_from_volume(p: torch.Tensor) -> torch.Tensor:
  """ This returns the cut out mesh when cutting an object with the silhouettes of 3d tensor p.
  """
  p_x = mellowmax(p, 0, keepdim=True) # 2D silhouette when looking along x-axis
  p_y = mellowmax(p, 1, keepdim=True)
  p_z = mellowmax(p, 2, keepdim=True)
  p_cut = p_x * p_y * p_z # condition for a voxel to not be removed
  return p_cut

# Optimization Loop

In [ ]:
U = V.clone() # deformed vertices
U.requires_grad_(True)
optim = torch.optim.Adam([U], lr=step)
_, init_loss_bandsaw, _ = loss_fn(U)
print(f"Initial bandsaw loss: {init_loss_bandsaw:.5f}).")

# optimize
itr = tqdm.trange(max_steps, desc=f"Loss: -")
try:
  for t in itr:
    # bookkeeping
    loss, loss_bandsaw, loss_deform = loss_fn(U)
    itr.set_description(f"Loss: {loss:.5f}")

    U_prev = U.detach().clone()
    optim.zero_grad()
    loss.backward()
    optim.step()

    converged = torch.linalg.vector_norm(U.detach()-U_prev, dim=1).max().item() < eps
    if converged:
      break
except KeyboardInterrupt:
  print(f"Skipping {max_steps - t} iterations.")

improved = 1 - loss_bandsaw/init_loss_bandsaw
print(f"Final loss: {loss:.5f} (Bandsaw: {loss_bandsaw:.5f}, Deform {loss_deform:.5f}). Relative cut improvement: {improved:.4f}.")

## Show results

In [ ]:
ps.init()

with torch.no_grad():
  voxel_saw = dvx.voxelize(d, U, F)
  voxel_orig = dvx.voxelize(d, V, F)
  voxel_saw_cut = cut_from_volume(voxel_saw)
  voxel_orig_cut = cut_from_volume(voxel_orig)

origin = [0., 0., 0.]
cell_width = 3*[1./(d-1)]

oc_saw_cut = (voxel_saw_cut > 1e-3).nonzero(as_tuple=False).numpy()
oc_orig_cut = (voxel_orig_cut > 1e-3).nonzero(as_tuple=False).numpy()
val_saw = voxel_saw_cut[oc_saw_cut[:, 0], oc_saw_cut[:, 1], oc_saw_cut[:, 2]].numpy()
val_orig = voxel_orig_cut[oc_orig_cut[:, 0], oc_orig_cut[:, 1], oc_orig_cut[:, 2]].numpy()

# register the grids
ps_grid_saw = ps.register_sparse_volume_grid("Optimized cut shape", origin, cell_width, oc_saw_cut)
ps_grid_orig = ps.register_sparse_volume_grid("Cut shape original", origin, cell_width, oc_orig_cut, enabled=False)
ps_grid_saw.add_scalar_quantity("filtered_w", val_saw, defined_on='cells', enabled=True)
ps_grid_orig.add_scalar_quantity("filtered_w", val_orig, defined_on='cells', enabled=True)

ps.show()